# Gradient Boosting — regresja krok po kroku

**Cel szkoleniowy:** pokazać ideę boostingową: kolejne słabe modele poprawiają błędy poprzednich.

W notebooku:
1. użyjemy problemu regresji,
2. zbudujemy model,
3. ocenimy MAE, RMSE i R²,
4. przeanalizujemy predykcje,
5. zobaczymy ważność cech,
6. sprawdzimy wpływ `learning_rate`, `n_estimators` i `max_depth`.

## 1. Intuicja

Gradient Boosting buduje modele **sekwencyjnie**.

Każde kolejne drzewo próbuje skorygować błędy wcześniejszego zespołu modeli.

Kluczowa zależność:
- mały `learning_rate` → zwykle potrzebujemy więcej drzew,
- duży `learning_rate` → szybsze uczenie, ale większe ryzyko przeuczenia.

Najważniejsze hiperparametry:
- `n_estimators`,
- `learning_rate`,
- `max_depth`,
- `min_samples_leaf`,
- `subsample`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42

## 2. Dane

Użyjemy wbudowanego zbioru **Diabetes**.

Zmienne wejściowe opisują cechy pacjentów, a celem jest przewidywanie ilościowej miary progresji choroby po roku.

In [ ]:
data = load_diabetes(as_frame=True)

X = data.data
y = data.target

print("Rozmiar X:", X.shape)
print("\nStatystyki zmiennej docelowej:")
print(y.describe())

X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

## 3. Model bazowy

In [ ]:
gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=2,
    random_state=RANDOM_STATE
)

gb_model.fit(X_train, y_train)
y_pred = gb_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE :", round(mae, 3))
print("RMSE:", round(rmse, 3))
print("R²  :", round(r2, 3))

### Jak interpretować metryki?

- **MAE** — średni bezwzględny błąd.
- **RMSE** — mocniej karze duże błędy.
- **R²** — jaka część zmienności celu jest wyjaśniana przez model.

## 4. Predykcja vs wartość rzeczywista

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, alpha=0.7)
min_v = min(y_test.min(), y_pred.min())
max_v = max(y_test.max(), y_pred.max())
plt.plot([min_v, max_v], [min_v, max_v])
plt.xlabel("Wartość rzeczywista")
plt.ylabel("Predykcja")
plt.title("Gradient Boosting — predykcja vs rzeczywistość")
plt.show()

## 5. Rozkład błędów

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(7, 5))
plt.hist(residuals, bins=20)
plt.xlabel("Błąd: y_true - y_pred")
plt.ylabel("Liczba obserwacji")
plt.title("Rozkład błędów")
plt.show()

pd.Series(residuals).describe()

## 6. Ważność cech

In [ ]:
importance = pd.Series(
    gb_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importance.sort_values().plot(kind="barh", figsize=(8, 5))
plt.title("Gradient Boosting — ważność cech")
plt.xlabel("Feature importance")
plt.show()

importance

## 7. Eksperyment hiperparametrów

Zbadamy wpływ:
- liczby drzew,
- learning rate,
- głębokości drzewa.

In [ ]:
results = []

for n_estimators in [50, 100, 200, 400]:
    for learning_rate in [0.01, 0.05, 0.1]:
        for max_depth in [1, 2, 3]:
            model = GradientBoostingRegressor(
                n_estimators=n_estimators,
                learning_rate=learning_rate,
                max_depth=max_depth,
                random_state=RANDOM_STATE
            )
            model.fit(X_train, y_train)
            pred = model.predict(X_test)

            results.append({
                "n_estimators": n_estimators,
                "learning_rate": learning_rate,
                "max_depth": max_depth,
                "MAE": mean_absolute_error(y_test, pred),
                "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
                "R2": r2_score(y_test, pred)
            })

results_df = pd.DataFrame(results).sort_values("RMSE")
results_df.head(12)

## 8. Staged prediction — jak model poprawia się wraz z kolejnymi drzewami?

To bardzo dobry fragment do pokazania na żywo.

In [ ]:
train_rmse = []
test_rmse = []

for train_pred, test_pred in zip(
    gb_model.staged_predict(X_train),
    gb_model.staged_predict(X_test)
):
    train_rmse.append(np.sqrt(mean_squared_error(y_train, train_pred)))
    test_rmse.append(np.sqrt(mean_squared_error(y_test, test_pred)))

plt.figure(figsize=(8, 5))
plt.plot(train_rmse, label="train RMSE")
plt.plot(test_rmse, label="test RMSE")
plt.xlabel("Liczba drzew")
plt.ylabel("RMSE")
plt.title("Uczenie Gradient Boosting")
plt.legend()
plt.show()

## 9. Opcje do pokazania na szkoleniu

Można zmieniać:
- `learning_rate=0.01 / 0.05 / 0.1 / 0.2`,
- `n_estimators=50 / 100 / 300 / 500`,
- `max_depth=1 / 2 / 3 / 5`,
- `subsample=0.5 / 0.8 / 1.0`.

### Ćwiczenie
1. Ustaw `learning_rate=0.2` i `n_estimators=50`.
2. Ustaw `learning_rate=0.01` i `n_estimators=500`.
3. Porównaj RMSE.
4. Sprawdź, przy ilu drzewach błąd testowy przestaje wyraźnie maleć.

### Wniosek
Gradient Boosting jest zwykle mocniejszy od pojedynczego drzewa, ale wymaga rozsądnego strojenia tempa uczenia i złożoności drzew.